# 01 — Cycle proie–prédateur (Lotka–Volterra) dans EcoSim

**Objectif** : reproduire qualitativement le cycle de Lotka–Volterra dans le moteur
EcoSim avec un triplet ressource / herbivore / carnivore (`Herbe → Lapin → Renard`).

Le modèle classique prédit, en l'absence de structure spatiale, des oscillations couplées :
le pic des prédateurs suit avec un déphasage celui des proies, et la trajectoire dans l'espace
(proie, prédateur) forme une orbite fermée. Dans un moteur agent-based avec saisons, énergie,
génétique et structure spatiale, on s'attend à des cycles *amortis* (l'amortissement
vient du caractère stochastique et de l'hétérogénéité du terrain).

## Reproductibilité

Le seed, la taille de grille, le nombre de ticks, et le preset de terrain sont fixés au début
du notebook (`CONFIG`). Toute exécution sur la même version d'EcoSim donnera exactement les
mêmes courbes. Le bloc final exporte la configuration en JSON à côté du notebook.

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ecosim.engine.api import SimConfig, Simulation
from ecosim.version import __version__ as ECOSIM_VERSION

# ── Configuration de l'expérience ──────────────────────────────────────────
CONFIG = {
    "ecosim_version":   ECOSIM_VERSION,
    "seed":             42,
    "grid_size":        150,
    "terrain_preset":   "temperate",
    "ticks":            6000,
    "sampling_every":   30,  # tick spacing for the time-series
    "species": {
        "Herbe":  {"count": 900},
        "Lapin":  {"count": 80},
        "Renard": {"count": 10},
    },
}
print(f"ecosim version : {ECOSIM_VERSION}")
print(f"seed           : {CONFIG['seed']}")
print(f"grid_size      : {CONFIG['grid_size']}")
print(f"ticks          : {CONFIG['ticks']}")

## 1. Construction de la simulation

On construit la simulation via `ecosim.engine.api.Simulation`. Les espèces sont lues depuis
les fichiers JSON livrés dans `ecosim/data/species/` (paquet installé). Les `count` du CONFIG
remplacent ceux des JSON pour avoir des populations initiales adaptées à la taille de grille.

In [ ]:
# Localise le dossier ecosim/data/species/ depuis le paquet installé.
import ecosim as _ecosim_pkg
SPECIES_DIR = Path(_ecosim_pkg.__file__).parent / "data" / "species"

def load_species_params(name: str) -> dict:
    spec = json.loads((SPECIES_DIR / f"{name.lower()}.json").read_text(encoding="utf-8"))
    params = spec.get("params", spec)
    if "color" in params:
        params["color"] = tuple(params["color"])
    return params

cfg = SimConfig(
    seed=CONFIG["seed"],
    grid_size=CONFIG["grid_size"],
    terrain_preset=CONFIG["terrain_preset"],
    out_path=None,  # pas de recorder .db pour ce notebook
)
sim = Simulation(cfg)
for name, opts in CONFIG["species"].items():
    sim.add_species(load_species_params(name), count=opts["count"])

print("Populations initiales :", sim.populations)

## 2. Exécution avec échantillonnage

On lance la simulation pour `CONFIG['ticks']` ticks. Toutes les `sampling_every` ticks, on
enregistre les comptes par espèce. Le callback `on_progress` du runner échantillonne aux
intervalles imposés par EngineRunner (~ticks/100), on filtre côté Python pour avoir un pas
régulier.

In [ ]:
series: dict[str, list[int]] = defaultdict(list)
ticks: list[int] = []

def _record(tick: int, counts: dict) -> None:
    ticks.append(tick)
    for name in CONFIG["species"]:
        series[name].append(counts.get(name, 0))

# État initial (tick 0 effectif après spawn)
_record(sim.tick, sim.populations)
summary = sim.run(CONFIG["ticks"], on_progress=_record)

print(f"Ticks exécutés : {summary.ticks_done}")
print(f"Temps écoulé   : {summary.elapsed_s:.1f} s ({summary.ticks_done/summary.elapsed_s:.1f} ticks/s)")
print(f"Populations finales : {summary.final_populations}")

## 3. Séries temporelles — proies / prédateurs / ressource

Trois axes côte-à-côte (ressource, proie, prédateur) pour ne pas écraser visuellement les
renards (effectifs ×10–×100 plus faibles que lapins ou herbe). On marque les pics et creux
principaux pour faire ressortir le déphasage attendu.

In [ ]:
t_arr = np.asarray(ticks)
herbe = np.asarray(series["Herbe"])
lapin = np.asarray(series["Lapin"])
renard = np.asarray(series["Renard"])

fig, axes = plt.subplots(3, 1, figsize=(9, 6.5), sharex=True)
axes[0].plot(t_arr, herbe, color="#2e8b57")
axes[0].set_ylabel("Herbe")
axes[0].set_title(f"EcoSim {ECOSIM_VERSION} — Cycle Lotka–Volterra (seed={CONFIG['seed']}, grille={CONFIG['grid_size']}²)")
axes[0].grid(alpha=0.3)

axes[1].plot(t_arr, lapin, color="#d6b85a")
axes[1].set_ylabel("Lapin")
axes[1].grid(alpha=0.3)

axes[2].plot(t_arr, renard, color="#b34727")
axes[2].set_ylabel("Renard")
axes[2].set_xlabel("Tick")
axes[2].grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 4. Portrait de phase proie ↔ prédateur

Trajectoire dans le plan (lapin, renard). En l'absence de stochasticité et de capacité de
charge, l'équation de Lotka–Volterra prédit des orbites *fermées* autour d'un point fixe. Ici,
on observe en général des cycles **amortis** : la trajectoire spirale vers un point
d'équilibre (ou s'effondre si une espèce s'éteint). La couleur encode le tick (clair = début,
foncé = fin).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
# Segments colorés par tick — du clair au foncé.
points = np.column_stack([lapin, renard])
segments = np.stack([points[:-1], points[1:]], axis=1)
from matplotlib.collections import LineCollection
lc = LineCollection(segments, cmap="viridis", linewidth=1.4)
lc.set_array(t_arr[:-1])
ax.add_collection(lc)
ax.set_xlim(lapin.min() - 5, lapin.max() + 5)
ax.set_ylim(renard.min() - 2, renard.max() + 2)
ax.set_xlabel("Lapins")
ax.set_ylabel("Renards")
ax.set_title("Portrait de phase")
ax.grid(alpha=0.3)
fig.colorbar(lc, ax=ax, label="Tick")
fig.tight_layout()
plt.show()

## 5. Mesure quantitative — autocorrélation et déphasage

On mesure :
- la période dominante des oscillations du lapin par autocorrélation,
- le déphasage proie/prédateur via inter-corrélation,

comme contrôle quantitatif du caractère oscillatoire.

In [ ]:
def _normalise(x: np.ndarray) -> np.ndarray:
    x = x.astype(float) - x.mean()
    std = x.std()
    return x / std if std > 0 else x

lap_n = _normalise(lapin)
ren_n = _normalise(renard)
n = len(lap_n)

if n > 4:
    acf = np.correlate(lap_n, lap_n, mode="full")[n - 1:]
    # première remontée après le premier zero crossing → période approximative
    dt = CONFIG["sampling_every"]
    period_ticks = None
    crossed = False
    for k in range(1, len(acf)):
        if not crossed and acf[k] < 0:
            crossed = True
            continue
        if crossed and acf[k] > 0 and (k == 1 or acf[k] > acf[k-1]):
            period_ticks = k * dt
            break
    xcorr = np.correlate(lap_n, ren_n, mode="full")
    lag = xcorr.argmax() - (n - 1)
    lag_ticks = lag * dt
    print(f"Période approchée des oscillations lapin : ~{period_ticks} ticks" if period_ticks else "Pas de période claire détectée")
    print(f"Déphasage proie/prédateur (lag du max de cross-corr) : {lag_ticks} ticks")
else:
    print("Pas assez d'échantillons pour autocorrélation.")

## 6. Sauvegarde de la configuration

On écrit `01_lotka_volterra.config.json` à côté du notebook avec la version EcoSim, le seed,
les paramètres de grille, et les compteurs d'espèces. Un autre utilisateur peut rejouer
exactement le même run en relisant ce JSON.

In [ ]:
CONFIG_OUT = Path("01_lotka_volterra.config.json")
CONFIG_OUT.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(f"Configuration écrite dans {CONFIG_OUT.resolve()}")

## Notes

- **Amortissement.** Dans un moteur agent-based avec capacité de charge spatiale, les
  oscillations s'amortissent plutôt qu'elles ne forment des orbites fermées. C'est cohérent
  avec la théorie de Rosenzweig–MacArthur. Pour reproduire des cycles soutenus, augmenter
  `max_population` ou désactiver la limite (mode `soft`).
- **Saisonnalité.** EcoSim applique un facteur saisonnier sur la croissance des plantes
  (`sin(2π·tick / SIM_YEAR)`). À `SIM_YEAR = 438 000` ticks, la saison ne tourne pas vraiment
  sur 6000 ticks — l'effet est négligeable ici.
- **Extinction des prédateurs.** Si les renards s'éteignent avant la fin, augmenter `count`
  initial, allonger la grille (`grid_size=200`), ou réduire `energy_consumption` dans
  `renard.json`.